## Practice 3 Hugging Face Transformers: Sentiment Analysis and Binary Text Classification

**Mục tiêu tổng quát:**  
Làm quen với hệ sinh thái Hugging Face Transformers thông qua hai bài tập:
- **Exercise 1:** Sử dụng model đã fine-tuned sẵn để thực hiện inference.
- **Exercise 2:** Fine-tune một pretrained model (generic) cho bài toán binary text classification.

### 1. Mục tiêu

- Phân biệt hai tầng kiến thức: **Pretraining** (language knowledge) và **Downstream Fine-tuning**
  (sentiment classification knowledge).

| Exercise | Mục tiêu chính | Có training không? | Checkpoint sử dụng |
|----------|----------------|--------------------|--------------------|
| **Exercise 1** | Inference + hiểu tokenizer | Không | `distilbert-base-uncased-finetuned-sst-2-english` |
| **Exercise 2** | Fine-tuning toàn bộ quy trình | Có | `distilbert-base-uncased` (generic) |


### (a) Vì sao chọn Rotten Tomatoes

| Tiêu chí | Rotten Tomatoes | IMDb |
|---|---|---|
| Labeled samples | 10,662 | 50,000 |
| Train split | 8,530 | 25,000 |
| Validation split | 1,066 | Không có split chính thức riêng |
| Test split | 1,066 | 25,000 |
| Binary sentiment | Có | Có |
| Độ dài văn bản điển hình | Ngắn | Dài hơn |
| Chi phí fine-tuning | Thấp hơn | Cao hơn |
| Phù hợp cho notebook thực hành gọn | Rất cao | Cao |
| Bám sát tutorial chính thức Hugging Face | Trung bình | Rất cao |

**Lý do chọn:** bài toán phân loại nhị phân, khối lượng tính toán vừa phải cho notebook thực hành, đã
có sẵn 3 split train/validation/test, review tương đối ngắn, tránh phải tự tạo thêm validation split,
hỗ trợ một quy trình thực nghiệm sạch trên máy cá nhân (CPU-only).

```
Train: 8,530
Validation: 1,066
Test: 1,066
Total: 10,662

Label 0: NEGATIVE
Label 1: POSITIVE
```


## (b) Vì sao chọn DistilBERT

```
flowchart LR
    A[Tokenized Text] --> B[DistilBERT Backbone]
    B --> C[Contextual Representation]
    C --> D[Classification Head]
    D --> E[Two Logits]
    E --> F[NEGATIVE or POSITIVE]
```

| Model | Ưu điểm chính | Hạn chế chính |
|---|---|---|
| DistilBERT | Nhẹ, thực tế | Capacity thấp hơn BERT-base |
| BERT-base | Baseline kinh điển của Transformer | Chi phí tính toán cao hơn |
| RoBERTa-base | Biểu diễn ngôn ngữ mạnh | Chi phí tính toán cao hơn |
| MiniLM | Rất nhẹ | Ít bám sát quy trình giới thiệu truyền thống |
| ALBERT | Hiệu quả về tham số | Đặc tính kiến trúc khác biệt |

**Lựa chọn triển khai chính: DistilBERT** - Transformer đã pretrained, nhẹ hơn BERT-base, hỗ trợ trực
tiếp sequence classification, phù hợp sentiment analysis tiếng Anh, giảm chi phí training nhưng vẫn giữ
nguyên bản chất transfer-learning workflow, phù hợp môi trường lab CPU-only.


### (c) Bản chất khái niệm Transfer Learning

```
flowchart TD
    A[Pretraining] --> B[General Language Knowledge]
    B --> C[Downstream Fine-Tuning]
    C --> D[Binary Sentiment Knowledge]
    D --> E[Inference]
    E --> F[Positive or Negative Prediction]
```

```
Exercise 1
Already fine-tuned model
        |
        v
Inference

Exercise 2
Generic pretrained model
        |
        v
Task-specific fine-tuning
        |
        v
Binary classifier
```


## (d) Fine-tuning khác Feature Extraction như thế nào

```
Feature extraction
Freeze Transformer backbone
        |
        v
Train only classifier

Fine-tuning
Update Transformer backbone
        +
Update classification head
```

| Tiêu chí | Feature Extraction | Fine-tuning                       |
|----------|--------------------|-----------------------------------|
| Trọng số base model | Freeze (không cập nhật) | Được cập nhật cùng classification head |
| Vai trò base model | Chỉ trích xuất đặc trưng cố định | Tự điều chỉnh biểu diễn cho phù hợp task |
| Chi phí tính toán | Thấp hơn | Cao hơn |
| Áp dụng trong bài này | Không dùng | Có (Trainer fine-tune toàn bộ model) |


Practice 3 (Exercise 2) áp dụng **Fine-tuning** toàn bộ backbone `distilbert-base-uncased` được cập
nhật cùng với classification head, không freeze layer nào.


## Notebook Architecture

```
flowchart TD
    P0[Phase 0<br/>Practice Overview] --> P1[Phase 1<br/>Environment and Reproducibility]
    P1 --> P2[Phase 2<br/>Pretrained Sentiment Inference]
    P2 --> P3[Phase 3<br/>Tokenization Investigation]
    P3 --> P4[Phase 4<br/>Dataset Loading]
    P4 --> P5[Phase 5<br/>EDA and Sanity Checks]
    P5 --> P6[Phase 6<br/>Tokenizer and Preprocessing]
    P6 --> P7[Phase 7<br/>Model Construction]
    P7 --> P8[Phase 8<br/>Metrics and Training Configuration]
    P8 --> P9[Phase 9<br/>Fine-Tuning]
    P9 --> P10[Phase 10<br/>Learning Curves]
    P10 --> P11[Phase 11<br/>Validation and Test Evaluation]
    P11 --> P12[Phase 12<br/>Error Analysis]
    P12 --> P13[Phase 13<br/>New-Sentence Inference]
    P13 --> P14[Phase 14<br/>Save and Reload]
    P14 --> P15[Phase 15<br/>Final Summary]
```





## Bản đồ tổng thể Pipeline End-to-End

```
flowchart TD
    S[START] --> A[Environment and Seeds]

    A --> B[Exercise 1]
    B --> C[Load Fine-Tuned Sentiment Model]
    C --> D[Inspect Sentence Tokens]
    D --> E[Run Sentiment Inference]

    E --> F[Exercise 2]
    F --> G[Load Rotten Tomatoes Dataset]
    G --> H[EDA and Data Quality Checks]
    H --> I[Load DistilBERT Tokenizer]
    I --> J[Token-Length Analysis]
    J --> K[Tokenize Dataset]
    K --> L[Dynamic Padding]
    L --> M[Load Pretrained DistilBERT Classifier]
    M --> N[Model Sanity Check]
    N --> O[Define Metrics]
    O --> P[Configure TrainingArguments]
    P --> Q[Create Trainer]
    Q --> R[Debug Subset Run]
    R --> T{Pipeline Valid?}
    T -- No --> U[Fix and Re-run]
    U --> R
    T -- Yes --> V[Full Fine-Tuning]
    V --> W[Validation Monitoring]
    W --> X[Load Best Checkpoint]
    X --> Y[Final Test Evaluation]
    Y --> Z[Confusion Matrix]
    Z --> AA[Error Analysis]
    AA --> AB[New-Sentence Inference]
    AB --> AC[Save Model and Tokenizer]
    AC --> AD[Reload Sanity Test]
    AD --> AE[FINAL SUMMARY]
```


## 5. Phạm vi Stage 1 

| Phase | Tên Phase | Mục tiêu chính |
|-------|-----------|----------------|
| 0 | Practice Overview | Định nghĩa bài toán + kiến trúc học thuật |
| 1 | Environment & Reproducibility | Setup môi trường, seed, device |
| 2 | Pretrained Sentiment Inference | Exercise 1 - Inference |
| 3 | Tokenization Investigation | Phân tích tokenizer |
| 4 | Dataset Loading | Load Rotten Tomatoes |
| 5 | Dataset EDA & Sanity Checks | Phân tích dữ liệu |
| 6 | Tokenizer & Preprocessing | Chuẩn bị data cho fine-tuning |
|   |                           |                                  |



## 6. Quyết định kỹ thuật quan trọng

| Hạng mục | Quyết định | Status |
|----------|----------|--------|
| Model Exercise 1 | `distilbert-base-uncased-finetuned-sst-2-english` | Approved |
| Model Exercise 2 | `distilbert-base-uncased` | Approved |
| Dataset | Rotten Tomatoes | Approved |
| Hardware | CPU only | Approved |
| Random Seed | 42 | Approved |

## 7. Final Conceptual Summary

```
flowchart LR
    A[General Language Pretraining] --> B[Pretrained DistilBERT]
    B --> C[Binary Sentiment Fine-Tuning]
    C --> D[Task-Specific Classifier]
    D --> E[Validation]
    E --> F[Independent Test Evaluation]
    F --> G[Reusable Sentiment Model]
```

```
Exercise 1
Pretrained model reuse
        +
Tokenizer understanding

Exercise 2
Transfer learning
        +
Fine-tuning
        +
Scientific evaluation
        +
Reusable model artifacts
```
